In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path


### Setup

In [ ]:
sns.set_context("paper", font_scale=2, rc={
    "font.size": 50,
    "axes.titlesize": 50,
    "axes.labelsize": 36,
    "xtick.labelsize": 36,
    "ytick.labelsize": 36,
    "legend.fontsize": 18,
    "legend.title_fontsize": 18,
})
sns.set_style("whitegrid")

In [ ]:
metrics2label = {
    'empirical': 'Empirical',
    'kl_backward': r'Rényi Divergence' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n' + r'$\mathit{(KL\ Divergence)}$',
    'renyi_divergence_backward_2': r'Rényi Divergence' + '\n' + r'$(\alpha = 2)$',
    'renyi_divergence_backward_3': r'Rényi Divergence' + '\n' + r'$(\alpha = 3)$',
    'renyi_divergence_backward_4': r'Rényi Divergence' + '\n' + r'$(\alpha = 4)$',
    'renyi_divergence_backward_5': r'Rényi Divergence' + '\n' + r'$(\alpha = 5)$',
    'renyi_divergence_backward_6': r'Rényi Divergence' + '\n' + r'$(\alpha = 6)$',
    'cross_entropy_backward': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n'  + r'$(\mathit{Shannon\ Cross\text{-}Entropy})$',
    'renyi_crossent_backward_2': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 2)$',
    'renyi_crossent_backward_3': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 3)$',
    'renyi_crossent_backward_4': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 4)$',
    'renyi_crossent_backward_5': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 5)$',
    'ccg_kl': 'Supertag KL Divergence' + '\n' + r'$\mathit{(Rényi\ Divergence\ } (\alpha \rightarrow 1))$',
    'synsurp': 'Supertag Surprisal',
    'gpt2_surp': 'Lexical Surprisal' + '\n' + '(GPT2-small)',
    'roberta_surp': 'Lexical Surprisal'
}

metric_categories = {
    'Empirical': ['empirical'],
    'Baselines': ['roberta_surp', 'synsurp'],
    'Syntactic Belief Update': ['kl_backward', 'renyi_divergence_backward_2', 'renyi_divergence_backward_3', 'renyi_divergence_backward_4', 'renyi_divergence_backward_5'],
}

label_categories = {k: [metrics2label[m] for m in v] for k, v in metric_categories.items()}

def add_labels_to_df(df, metric_col='model'):
    df['model_label'] = [metrics2label[m] for m in df[metric_col]]


## EOIs


## From BRM

In [ ]:
fillers = pd.read_csv('r_analyses/results/EOIs/brm/final_filler_EOIs.csv')
roi = pd.read_csv('r_analyses/results/EOIs/brm/final_roi_EOIs.csv')
fillers['model'] = (fillers['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)
add_labels_to_df(fillers)


roi['model'] = (roi['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)
add_labels_to_df(roi)



### Code for three-panel over either fillers or rois

In [ ]:
df = fillers
fig, ax = plt.subplots(
    nrows=1, 
    ncols=3,
    figsize=(55, 8),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 2, 5]}
)

melted = df.melt(
    id_vars=['model', 'model_label'],
    value_vars=['NPS', 'NPZ', 'MVRR'],
    var_name='type',
    value_name='metric' 
)

ci_lows = []
ci_highs = []
for i, melted_row in melted.iterrows():
    model = melted_row['model']
    gp_type = melted_row['type']
    CI_row = df[df['model'] == model]
    ci_lows.append(CI_row[f'{gp_type}_CI_low'].iloc[0]) 
    ci_highs.append(CI_row[f'{gp_type}_CI_high'].iloc[0]) 
melted['CI_low'] = ci_lows
melted['CI_high'] = ci_highs

hue_order = ['NPS', 'NPZ', 'MVRR']

for i, category in enumerate(metric_categories.keys()):
    curr_metrics = metric_categories[category]
    x_order = [metrics2label[metric] for metric in curr_metrics]
    curr_data = melted[melted['model'].isin(curr_metrics)]

    sns.barplot(
        data=curr_data,
        x='model_label',
        y='metric',
        order=x_order,
        hue='type',
        hue_order=hue_order,
        ax=ax[i],
        # dodge=True
    )
    if i != 2:
        ax[i].get_legend().remove()
    ax[i].set_title(category)
    ci_lookup = {
        (row['model_label'], row['type']): (row['CI_low'], row['CI_high'])
        for _, row in curr_data.iterrows()
    }

    # Seaborn lays out patches as: for each hue group, all x positions in order
    # So patch index = hue_idx * len(x_order) + x_idx
    patches = [p for p in ax[i].patches if p.get_width() > 0]  # skip legend patches

    for hue_idx, hue_val in enumerate(hue_order):
        for x_idx, x_label in enumerate(x_order):
            patch_idx = hue_idx * len(x_order) + x_idx
            if patch_idx >= len(patches):
                continue

            patch = patches[patch_idx]
            key = (x_label, hue_val)
            if key not in ci_lookup:
                continue

            ci_low, ci_high = ci_lookup[key]
            x = patch.get_x() + 0.5 * patch.get_width()
            y = patch.get_height()

            ax[i].errorbar(
                x, y,
                yerr=[[y - ci_low], [ci_high - y]],  # asymmetric: distance from bar top
                fmt='none',
                color='black',
                capsize=4,
                linewidth=1.5
            )
            

handles, labels = ax[2].get_legend_handles_labels()
ax[2].get_legend().remove()
fig.legend(
    handles, 
    ['NP/S', 'NP/Z', 'MV/RR'],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.3),
    fontsize=28,
    ncol=3,
    title='Garden Path Type',
    title_fontsize=30,
    frameon=False
)

ax[0].set_xlabel('')
ax[1].set_xlabel('')
ax[2].set_xlabel('')
ax[0].set_ylabel('Garden Path Effect (ms)', fontsize=36)
# fig.suptitle('Empirical and Predicted Slowdowns\n (Fit On Garden Paths Only)', fontsize=36, y=1.05)
plt.savefig('figures_for_paper/filler_eois.pdf', bbox_inches="tight", dpi=300)

### Code for one panel for gold

In [ ]:
gold = pd.read_csv('r_analyses/results/EOIs/gold/eois_gold.csv')
gold['model'] = (gold['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)
add_labels_to_df(gold)

gold_plot_categories = {
    'Empirical': ['empirical'],
    'Syntactic\n Belief Update': ['kl_backward'] #, 'renyi_divergence_backward_5', 'renyi_divergence_backward_6']
}

In [ ]:
melted = gold.melt(
    id_vars=['model', 'model_label'],
    value_vars=['NPS', 'NPZ', 'MVRR'],
    var_name='type',
    value_name='metric' 
)
ci_lows = []
ci_highs = []
for i, melted_row in melted.iterrows():
    model = melted_row['model']
    gp_type = melted_row['type']
    CI_row = gold[gold['model'] == model]
    ci_lows.append(CI_row[f'{gp_type}_CI_low'].iloc[0]) 
    ci_highs.append(CI_row[f'{gp_type}_CI_high'].iloc[0]) 
melted['CI_low'] = ci_lows
melted['CI_high'] = ci_highs

fig, ax = plt.subplots(
    nrows=1, 
    ncols=2,
    figsize=(24, 12),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 1]}
)


hue_order = ['NPS', 'NPZ', 'MVRR']

for i, category in enumerate(gold_plot_categories.keys()):
    curr_metrics = gold_plot_categories[category]
    x_order = [metrics2label[metric] for metric in curr_metrics]
    curr_data = melted[melted['model'].isin(curr_metrics)]

    sns.barplot(
        data=curr_data,
        x='model_label',
        y='metric',
        order=x_order,
        hue='type',
        hue_order=hue_order,
        ax=ax[i],
        dodge=True
    )
    ax[i].get_legend().remove()
    ax[i].set_title(category)
    ci_lookup = {
        (row['model_label'], row['type']): (row['CI_low'], row['CI_high'])
        for _, row in curr_data.iterrows()
    }

    # Seaborn lays out patches as: for each hue group, all x positions in order
    # So patch index = hue_idx * len(x_order) + x_idx
    patches = [p for p in ax[i].patches if p.get_width() > 0]  # skip legend patches

    for hue_idx, hue_val in enumerate(hue_order):
        for x_idx, x_label in enumerate(x_order):
            patch_idx = hue_idx * len(x_order) + x_idx
            if patch_idx >= len(patches):
                continue

            patch = patches[patch_idx]
            key = (x_label, hue_val)
            if key not in ci_lookup:
                continue

            ci_low, ci_high = ci_lookup[key]
            x = patch.get_x() + 0.5 * patch.get_width()
            y = patch.get_height()

            ax[i].errorbar(
                x, y,
                yerr=[[y - ci_low], [ci_high - y]],  # asymmetric: distance from bar top
                fmt='none',
                color='black',
                capsize=4,
                linewidth=1.5
            )

ax[0].set_xlabel('')

ax[1].set_xlabel('')
ax[0].set_ylabel('Garden Path Effect (ms)', fontsize=24)
# fig.suptitle('Empirical and Predicted Slowdown \n Against Belief Update to True Post-critical Structure', fontsize=36, y=1.04)
plt.savefig('figures_for_paper/eois_gold.pdf', bbox_inches='tight', dpi=300)

## Correlations Plot

In [ ]:
correlations = pd.read_csv('r_analyses/results/correlations/filler/final_corr.csv')
correlations['model'] = (correlations['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)

In [ ]:
correlations[(correlations['model']=='empirical') & (correlations['type']=='aggregate')]['correlation'].mean()

### Aggregate

In [ ]:
model_order = ['empirical'] + metric_categories['Baselines'] + metric_categories['Syntactic Belief Update']
hues = {}
hues['empirical'] = 'grey'
for i, model in enumerate(metric_categories['Baselines']):
    hues[model] = sns.color_palette('Blues', len(metric_categories)+2)[i]

for i, model in enumerate(metric_categories['Syntactic Belief Update']):
    hues[model] = sns.color_palette('Greens', len(metric_categories)+2)[i]


filtered = correlations[correlations['type'] == 'aggregate'].copy()
filtered['model_label'] = filtered['model'].map(metrics2label)


# Convert model_order to label space
model_order_labels = [metrics2label[m] for m in model_order]

# Convert palette to label space
hues_labelspace = {metrics2label[k]: v for k, v in hues.items()}

fig, ax = plt.subplots(
    nrows=1, 
    ncols=3,
    figsize=(55, 6),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 2, 5]}
) 
ax[0].set_ylim(0,1)

# Empirical
data = filtered[filtered['model'].isin(metric_categories['Empirical'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Empirical']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Empirical']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[0]
)
sns.swarmplot(
    data=data.sample(n=500),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[0]
)

# Baseline
data = filtered[filtered['model'].isin(metric_categories['Baselines'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Baselines']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Baselines']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[1]
)
sns.swarmplot(
    data=data.sample(n=1000),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[1]
)


# Parser
data = filtered[filtered['model'].isin(metric_categories['Syntactic Belief Update'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Syntactic Belief Update']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Syntactic Belief Update']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[2]
)
sns.swarmplot(
    data=data.sample(n=1000),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[2]
)


# plt.tight_layout()
ax[0].set_ylabel("Correlation", fontsize=50)
ax[0].set_title('Empirical')
ax[0].set_xlabel('')
ax[1].set_title('Baselines')
ax[1].set_xlabel('')
ax[2].set_title('Syntactic Belief Update')
ax[2].set_xlabel('')
plt.savefig('figures_for_paper/aggregated_item_correlations_filler.pdf', bbox_inches='tight', dpi=300)

### By Construction

In [ ]:
from matplotlib.patches import Patch
palette = {metrics2label[m]: hue for m, hue in hues.items()}
legend_elements = [Patch(facecolor=color, label=label) for label, color in palette.items()]
fig, ax = plt.subplots(figsize=(12, 6))
correlations['model_label'] = [metrics2label[m] for m in correlations['model']]
correlations = correlations[correlations['type'] != 'aggregate']
type_order = correlations['type'].unique().tolist()
sns.boxplot(
    data=correlations,
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    legend=False,
    ax=ax
)
sns.swarmplot(
    data=correlations.groupby(['type', 'model_label']).sample(n=100),
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    dodge=True,
    size=4,
    legend=False,
    ax=ax,
)
plt.tight_layout()
ax.set_ylim(-0.5, 1)
ax.set_ylabel('Correlation')
ax.set_xlabel('Garden Path Type')
baseline_labels = [metrics2label[m] for m in model_order if m in metric_categories['Baselines']]
divergence_labels = [metrics2label[m] for m in model_order if m in metric_categories['Syntactic Belief Update']]
empirical_elements = [Patch(facecolor=palette['Empirical'], label='Empirical')]
baseline_elements = [Patch(facecolor=palette[l], label=l) for l in baseline_labels]
divergence_elements = [Patch(facecolor=palette[l], label=' '.join(l.split('\n'))) for l in divergence_labels]
divergence_elements[0].set_label('Rényi Divergence ' + r'$(\alpha \rightarrow 1)$')

# No legends on ax — save the main figure clean
plt.savefig('figures_for_paper/filler_itemwise_by_type.pdf', bbox_inches='tight', dpi=300)

# Standalone legend figure
fig_leg, ax_leg = plt.subplots(figsize=(3, 4))
ax_leg.axis('off')

leg0 = ax_leg.legend(
    handles=empirical_elements,
    title='Empirical',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 1.0),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg0.get_title().set_fontweight('bold')
ax_leg.add_artist(leg0)

leg1 = ax_leg.legend(
    handles=baseline_elements,
    title='Baselines',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 0.88),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg1.get_title().set_fontweight('bold')
ax_leg.add_artist(leg1)

leg2 = ax_leg.legend(
    handles=divergence_elements,
    title='Syntactic Belief Update',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 0.7),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg2.get_title().set_fontweight('bold')

fig_leg.savefig('figures_for_paper/legend_for_corrs.pdf', bbox_inches='tight', dpi=300)

## Gold

In [ ]:
correlations = pd.read_csv('r_analyses/results/correlations/gold/correlations.csv')
correlations['model'] = (correlations['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)

In [ ]:

correlations.head(10)

In [ ]:
model_order = gold_plot_categories['Empirical'] + gold_plot_categories['Syntactic\n Belief Update']
hues = {}
hues['empirical'] = 'grey'

for i, model in enumerate(gold_plot_categories['Syntactic\n Belief Update']):
    hues[model] = sns.color_palette('Greens', len(metric_categories)+2)[i]

filtered = correlations[correlations['type'] == 'aggregate'].copy()
filtered['model_label'] = filtered['model'].map(metrics2label)

# Convert palette to label space
hues_labelspace = {metrics2label[k]: v for k, v in hues.items()}
fig, ax = plt.subplots(
    nrows=1, 
    ncols=2,
    figsize=(24, 12),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 1]}
) 

# Empirical
data = filtered[filtered['model'].isin(gold_plot_categories['Empirical'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in gold_plot_categories['Empirical']]
baseline_palette = {metrics2label[m]: hues[m] for m in gold_plot_categories['Empirical']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[0]
)
sns.swarmplot(
    data=data.sample(n=500),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[0]
)

# Parser
data = filtered[filtered['model'].isin(gold_plot_categories['Syntactic\n Belief Update'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in gold_plot_categories['Syntactic\n Belief Update']]
baseline_palette = {metrics2label[m]: hues[m] for m in gold_plot_categories['Syntactic\n Belief Update']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[1]
)
sns.swarmplot(
    data=data.sample(500),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[1]
)



# plt.tight_layout()
ax[0].set_ylim([0, 1])
ax[0].set_ylabel("Correlation")
ax[0].set_title('Empirical')
ax[0].set_xlabel('')
ax[1].set_title('Syntactic\n Belief Update')
ax[1].set_xlabel('')
plt.subplots_adjust(top=0.85) 
plt.savefig('figures_for_paper/aggregated_item_correlations_gold.pdf', bbox_inches='tight', dpi=300)

In [ ]:
from matplotlib.patches import Patch

palette = {metrics2label[m]: hue for m, hue in hues.items()}
legend_elements = [Patch(facecolor=color, label=label) for label, color in palette.items()]


fig, ax = plt.subplots(figsize=(12, 6))
correlations['model_label'] = [metrics2label[m] for m in correlations['model']]
correlations = correlations[correlations['type'] != 'aggregate']
type_order = correlations['type'].unique().tolist()


sns.boxplot(
    data=correlations,
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    legend=False,
    ax=ax
)
sns.swarmplot(
    data=correlations.groupby(['type', 'model_label']).sample(n=100),
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    dodge=True,
    size=4,
    legend=False,
    ax=ax,
)


plt.tight_layout() 
# plt.title("Itemwise Correlations by Type\n (Computed Against True Post-Critical Structure)")
ax.set_ylabel('Correlation')
ax.set_xlabel('Garden Path Type')
plt.savefig('figures_for_paper/filler_itemwise_gold.pdf', bbox_inches='tight', dpi=300)

In [ ]:
correlations

# Supplementary Figures 

## Frequentist Renyi Alpha Sweep

In [ ]:
eois = pd.read_csv('alpha_sweep_data/roi_EOIs.csv')
eois = eois.melt(
    id_vars='model',
    value_vars=['NPS', 'NPZ', 'MVRR'],
)

renyi_metrics = {
    'empirical': 'Empirical',
    'kl_backward': r'Rényi Divergence' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n' + r'$\mathit{(KL\ Divergence)}$',
    'renyi_divergence_backward_2': r'Rényi Divergence' + '\n' + r'$(\alpha = 2)$',
    'renyi_divergence_backward_3': r'Rényi Divergence' + '\n' + r'$(\alpha = 3)$',
    'renyi_divergence_backward_4': r'Rényi Divergence' + '\n' + r'$(\alpha = 4)$',
    'renyi_divergence_backward_5': r'Rényi Divergence' + '\n' + r'$(\alpha = 5)$',
    'renyi_divergence_backward_6': r'Rényi Divergence' + '\n' + r'$(\alpha = 6)$',
    'cross_entropy_backward': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n',
    'renyi_crossent_backward_2': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 2)$',
    'renyi_crossent_backward_3': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 3)$',
    'renyi_crossent_backward_4': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 4)$',
    'renyi_crossent_backward_5': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 5)$',
}

eois = eois[eois['model'].isin(renyi_metrics.keys())]
add_labels_to_df(eois)


In [ ]:
fig, ax = plt.subplots(
    nrows=1, 
    ncols=3,
    figsize=(60, 8),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 6, 5]}
)
categories = {
    'Empirical': ['empirical'],
    'Rényi Divergence': ['kl_backward', 'renyi_divergence_backward_2', 'renyi_divergence_backward_3', 'renyi_divergence_backward_4', 'renyi_divergence_backward_5', 'renyi_divergence_backward_6'],
    'Rényi Cross-Entropy': ['cross_entropy_backward', 'renyi_crossent_backward_2', 'renyi_crossent_backward_3', 'renyi_crossent_backward_4', 'renyi_crossent_backward_5']
}

eois['model_label'] =['\n'.join(m.split('\n')[1:]) for m in eois['model_label']]

for i, (category, metrics) in enumerate(categories.items()):
    filtered = eois[eois['model'].isin(metrics)]
    sns.barplot(
        data=filtered,
        x='model_label',
        y='value',
        hue='variable',
        ax=ax[i],
    )
    ax[i].set_title(category)
    if i == 0:
        ax[i].set_ylabel('Garden Path Effect (ms)')
    if i != 2:
        ax[i].get_legend().remove()
    if i == 2:
        handles, labels = ax[i].get_legend_handles_labels()
        ax[i].legend(title='Garden Path Type', handles=handles, labels=['NP/S', 'NP/Z', 'MV/RR'])

    ax[i].set_xlabel('')


plt.savefig('figures_for_paper/frequentist_alpha_sweep_eois.pdf')
# fig.suptitle('Garden Path Effect (ms)')


In [ ]:
filtered

In [ ]:
corrs = pd.read_csv('alpha_sweep_data/lm_item_level_correlations.csv')
corrs = corrs[corrs['model'].isin(renyi_metrics.keys())]
corrs = corrs[corrs['type'] == 'aggregate']
add_labels_to_df(corrs)

In [ ]:
corrs

In [ ]:
fig, ax = plt.subplots(
    nrows=1, 
    ncols=2,
    figsize=(60, 8),
    sharey=True,
    gridspec_kw={'width_ratios': [6, 4]}
)

categories = {
    'Cross-Construction Correlation \n(Rényi Divergence)': ['kl_backward', 'renyi_divergence_backward_2', 'renyi_divergence_backward_3', 'renyi_divergence_backward_4', 'renyi_divergence_backward_5', 'renyi_divergence_backward_6'],
    'Cross-Construction Correlation \n(Rényi Cross-Entropy)': ['cross_entropy_backward', 'renyi_crossent_backward_2', 'renyi_crossent_backward_3', 'renyi_crossent_backward_4', 'renyi_crossent_backward_5']
}

corrs['model_label'] =['\n'.join(m.split('\n')[1:]) for m in corrs['model_label']]

for i, (category, metrics) in enumerate(categories.items()):
    filtered = corrs[corrs['model'].isin(metrics)]
    ax[i].set_title(category)
    sns.lineplot(
        data=filtered,
        x='model_label',
        y='correlation',
        ax=ax[i],
        marker='o',
        markersize=25,
        linewidth=10,
        color='green' if i == 0 else 'purple'
    )
    ax[i].set_xlabel('')
    # if i != 3:
    #     ax[i].get_legend().remove()
ax[0].set_ylabel('Correlation')
plt.savefig('figures_for_paper/frequentist_alpha_sweep_corrs.pdf')
# fig.suptitle('Cross-Construction Correlation')

## Other Baselines

In [ ]:
metrics2label = {
    'empirical': 'Empirical',
    'kl_backward': r'Rényi Divergence' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n' + r'$\mathit{(KL\ Divergence)}$',
    'renyi_divergence_backward_2': r'Rényi Divergence' + '\n' + r'$(\alpha = 2)$',
    'renyi_divergence_backward_3': r'Rényi Divergence' + '\n' + r'$(\alpha = 3)$',
    'renyi_divergence_backward_4': r'Rényi Divergence' + '\n' + r'$(\alpha = 4)$',
    'renyi_divergence_backward_5': r'Rényi Divergence' + '\n' + r'$(\alpha = 5)$',
    'renyi_divergence_backward_6': r'Rényi Divergence' + '\n' + r'$(\alpha = 6)$',
    'cross_entropy_backward': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha \rightarrow 1)$' + '\n'  + r'$(\mathit{Shannon\ Cross\text{-}Entropy})$',
    'renyi_crossent_backward_2': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 2)$',
    'renyi_crossent_backward_3': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 3)$',
    'renyi_crossent_backward_4': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 4)$',
    'renyi_crossent_backward_5': r'Rényi Cross-Entropy' + '\n' + r'$(\alpha = 5)$',
    'ccg_kl': 'Supertag KL Divergence',
    'synsurp': 'Supertag Surprisal',
    'gpt2_surp': 'Lexical Surprisal' + '\n' + '(GPT2-small)',
    'roberta_surp': 'Lexical Surprisal' + '\n' + '(Causal RoBERTa)',
}


### EOIs

In [ ]:
fillers = pd.read_csv('r_analyses/results/EOIs/brm/final_fillersupp_EOIs.csv')
roi = pd.read_csv('r_analyses/results/EOIs/brm/final_roisupp_EOIs.csv')
fillers['model'] = (fillers['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)
add_labels_to_df(fillers)


roi['model'] = (roi['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)
add_labels_to_df(roi)



In [ ]:
df = roi
fig, ax = plt.subplots(
    nrows=1, 
    ncols=3,
    figsize=(55, 8),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 2, 2]}
)

melted = df.melt(
    id_vars=['model', 'model_label'],
    value_vars=['NPS', 'NPZ', 'MVRR'],
    var_name='type',
    value_name='metric' 
)

ci_lows = []
ci_highs = []
for i, melted_row in melted.iterrows():
    model = melted_row['model']
    gp_type = melted_row['type']
    CI_row = df[df['model'] == model]
    ci_lows.append(CI_row[f'{gp_type}_CI_low'].iloc[0]) 
    ci_highs.append(CI_row[f'{gp_type}_CI_high'].iloc[0]) 
melted['CI_low'] = ci_lows
melted['CI_high'] = ci_highs

hue_order = ['NPS', 'NPZ', 'MVRR']
metric_categories = {
    'Empirical': ['empirical'],
    'Lexical Baselines': ['gpt2_surp', 'roberta_surp'],
    'Supertag Baselines': ['synsurp', 'ccg_kl']
}

for i, category in enumerate(metric_categories.keys()):
    curr_metrics = metric_categories[category]
    x_order = [metrics2label[metric] for metric in curr_metrics]
    curr_data = melted[melted['model'].isin(curr_metrics)]

    sns.barplot(
        data=curr_data,
        x='model_label',
        y='metric',
        order=x_order,
        hue='type',
        hue_order=hue_order,
        ax=ax[i],
        # dodge=True
    )
    if i != 2:
        ax[i].get_legend().remove()
    ax[i].set_title(category)
    ci_lookup = {
        (row['model_label'], row['type']): (row['CI_low'], row['CI_high'])
        for _, row in curr_data.iterrows()
    }

    # Seaborn lays out patches as: for each hue group, all x positions in order
    # So patch index = hue_idx * len(x_order) + x_idx
    patches = [p for p in ax[i].patches if p.get_width() > 0]  # skip legend patches

    for hue_idx, hue_val in enumerate(hue_order):
        for x_idx, x_label in enumerate(x_order):
            patch_idx = hue_idx * len(x_order) + x_idx
            if patch_idx >= len(patches):
                continue

            patch = patches[patch_idx]
            key = (x_label, hue_val)
            if key not in ci_lookup:
                continue

            ci_low, ci_high = ci_lookup[key]
            x = patch.get_x() + 0.5 * patch.get_width()
            y = patch.get_height()

            ax[i].errorbar(
                x, y,
                yerr=[[y - ci_low], [ci_high - y]],  # asymmetric: distance from bar top
                fmt='none',
                color='black',
                capsize=4,
                linewidth=1.5
            )
            

handles, labels = ax[1].get_legend_handles_labels()
ax[2].get_legend().remove()
# fig.legend(
#     handles, 
#     ['NP/S', 'NP/Z', 'MV/RR'],
#     loc='lower center',
#     bbox_to_anchor=(0.5, -0.3),
#     fontsize=28,
#     ncol=3,
#     title='Garden Path Type',
#     title_fontsize=30,
#     frameon=False
# )

ax[0].set_xlabel('')
ax[1].set_xlabel('')
ax[2].set_xlabel('')
ax[0].set_ylabel('Garden Path Effect (ms)', fontsize=36)
plt.savefig('figures_for_paper/roi_eois_baselines.pdf', bbox_inches="tight", dpi=300)

## Correlations

In [ ]:
correlations = pd.read_csv('r_analyses/results/correlations/roi/final_sup_corr.csv')
correlations['model'] = (correlations['model']
    .astype(str).str.split('.', n=1).str[0].str.split('_')
    .apply(lambda parts: '_'.join(parts[1:]) if len(parts) > 1 else parts[0])
)

In [ ]:
correlations

In [ ]:
correlations[(correlations['model']=='empirical') & (correlations['type']=='aggregate')]['correlation'].mean()

### Aggregate

In [ ]:
model_order = ['empirical'] + metric_categories['Lexical Baselines'] + metric_categories['Supertag Baselines']
hues = {}
hues['empirical'] = 'grey'
for i, model in enumerate(metric_categories['Lexical Baselines']):
    hues[model] = sns.color_palette('Blues', len(metric_categories)+2)[i]

for i, model in enumerate(metric_categories['Supertag Baselines']):
    hues[model] = sns.color_palette('Purples', len(metric_categories)+2)[i]


filtered = correlations[correlations['type'] == 'aggregate'].copy()
filtered['model_label'] = filtered['model'].map(metrics2label)


# Convert model_order to label space
model_order_labels = [metrics2label[m] for m in model_order]

# Convert palette to label space
hues_labelspace = {metrics2label[k]: v for k, v in hues.items()}

fig, ax = plt.subplots(
    nrows=1, 
    ncols=3,
    figsize=(55, 6),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 2, 2]}
) 
ax[0].set_ylim(0,1)

# Empirical
data = filtered[filtered['model'].isin(metric_categories['Empirical'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Empirical']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Empirical']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[0]
)
sns.swarmplot(
    data=data.sample(n=500),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[0]
)

# Baseline
data = filtered[filtered['model'].isin(metric_categories['Lexical Baselines'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Lexical Baselines']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Lexical Baselines']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[1]
)
sns.swarmplot(
    data=data.sample(n=1000),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[1]
)


# Parser
data = filtered[filtered['model'].isin(metric_categories['Supertag Baselines'])].copy()
data['model_label'] = data['model'].map(metrics2label)

baseline_labels = [metrics2label[m] for m in metric_categories['Supertag Baselines']]
baseline_palette = {metrics2label[m]: hues[m] for m in metric_categories['Supertag Baselines']}

sns.boxplot(
    data=data,
    x='model_label',
    y='correlation',
    order=baseline_labels,
    width=0.6,
    palette=baseline_palette,
    ax=ax[2]
)
sns.swarmplot(
    data=data.sample(n=1000),
    x='model_label',
    y='correlation',
    order=baseline_labels,
    palette=baseline_palette,
    ax=ax[2]
)


# plt.tight_layout()
ax[0].set_ylabel("Correlation", fontsize=50)
ax[0].set_title('Empirical')
ax[0].set_xlabel('')
ax[1].set_title('Baselines')
ax[1].set_xlabel('')
ax[2].set_title('Syntactic Belief Update')
ax[2].set_xlabel('')
plt.savefig('figures_for_paper/aggregated_item_correlations_roi_baselines.pdf', bbox_inches='tight', dpi=300)

### By Construction

In [ ]:
from matplotlib.patches import Patch
palette = {metrics2label[m]: hue for m, hue in hues.items()}
legend_elements = [Patch(facecolor=color, label=label) for label, color in palette.items()]
fig, ax = plt.subplots(figsize=(12, 6))
correlations['model_label'] = [metrics2label[m] for m in correlations['model']]
correlations = correlations[correlations['type'] != 'aggregate']
type_order = correlations['type'].unique().tolist()
sns.boxplot(
    data=correlations,
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    legend=False,
    ax=ax
)
sns.swarmplot(
    data=correlations.groupby(['type', 'model_label']).sample(n=100),
    x='type', y='correlation',
    hue='model_label',
    order=type_order,
    hue_order=[metrics2label[m] for m in model_order],
    palette=palette,
    dodge=True,
    size=4,
    legend=False,
    ax=ax,
)
plt.tight_layout()
ax.set_ylim(-0.5, 1)
ax.set_ylabel('Correlation')
ax.set_xlabel('Garden Path Type')
baseline_labels = [metrics2label[m] for m in model_order if m in metric_categories['Lexical Baselines']]
divergence_labels = [metrics2label[m] for m in model_order if m in metric_categories['Supertag Baselines']]
empirical_elements = [Patch(facecolor=palette['Empirical'], label='Empirical')]
baseline_elements = [Patch(facecolor=palette[l], label=l) for l in baseline_labels]
divergence_elements = [Patch(facecolor=palette[l], label=' '.join(l.split('\n'))) for l in divergence_labels]
# divergence_elements[0].set_label('Rényi Divergence ' + r'$(\alpha \rightarrow 1)$')

# No legends on ax — save the main figure clean
plt.savefig('figures_for_paper/roi_itemwise_by_type_supp.pdf', bbox_inches='tight', dpi=300)

# Standalone legend figure
fig_leg, ax_leg = plt.subplots(figsize=(3, 4))
ax_leg.axis('off')

leg0 = ax_leg.legend(
    handles=empirical_elements,
    title='Empirical',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 1.0),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg0.get_title().set_fontweight('bold')
ax_leg.add_artist(leg0)

leg1 = ax_leg.legend(
    handles=baseline_elements,
    title='Lexical Baselines',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 0.87),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg1.get_title().set_fontweight('bold')
ax_leg.add_artist(leg1)

leg2 = ax_leg.legend(
    handles=divergence_elements,
    title='Supertag Baselines',
    fontsize=9, title_fontsize=10,
    loc='upper left', bbox_to_anchor=(0, 0.59),
    ncols=1, frameon=False, handlelength=1.5, alignment='left'
)
leg2.get_title().set_fontweight('bold')

fig_leg.savefig('figures_for_paper/legend_for_corrs_supp.pdf', bbox_inches='tight', dpi=300)